## RDKit Fingerprints

https://www.rdkit.org/docs/GettingStartedInPython.html

In [8]:
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import numpy as np

In [9]:
with open("../data/01-result/drugs_df.pkl","rb") as f:
    drugs_df =pickle.load(f)

In [10]:
fpgen = AllChem.GetMorganGenerator(radius=2)
mol = Chem.MolFromSmiles('Cc1ccccc1')
fp = fpgen.GetFingerprint(mol)
bitstring = fp.ToBitString()  
arr = np.fromiter(bitstring, dtype=int)
arr

array([0, 0, 0, ..., 0, 0, 0], shape=(2048,))

In [11]:
drugs_df = drugs_df[ drugs_df["canonical_smiles"].notna() ]

In [12]:
# Source - https://stackoverflow.com/a
# Posted by let me down slowly
# Retrieved 2025-11-21, License - CC BY-SA 4.0

smiles_list = drugs_df["canonical_smiles"]#["O=C(NCc1cc(OC)c(O)cc1)CCCC/C=C/C(C)C", "CC(C)CCCCCC(=O)NCC1=CC(=C(C=C1)O)OC", "c1(C=O)cc(OC)c(O)cc1"]

# create a list of mols
mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

# create a list of fingerprints from mols
fps = np.array([np.fromiter(Chem.RDKFingerprint(mol).ToBitString(), dtype=int)  for mol in mols])


In [13]:
from sklearn.decomposition import PCA
import pandas as pd

pca = PCA(n_components=100)
X_reduced = pca.fit_transform(fps)
fp_df = pd.DataFrame(X_reduced, columns=[f'FP_{i+1}' for i in range(X_reduced.shape[1])])

print(X_reduced.shape)  # (num_molecules, 10)

(1547, 100)


In [14]:
fingerprints_df = pd.concat([drugs_df["drug_id"].reset_index(drop=True), fp_df], axis=1)
with open("../data/01-result/fingerprints_df.pkl","wb") as f:
    pickle.dump(fingerprints_df,f)